# VDA-HyperQuant: Colab Evaluation
This notebook demonstrates how to load Video-Depth-Anything (VDA), apply the **VDA-HyperQuant** model surgery, and run evaluation on sample video frames.

### Step 1: Clone the Repository and Install Dependencies

In [ ]:
# Clone the vdaquant repo
!git clone https://github.com/anrd30/vdaquant.git
%cd vdaquant

# Install dependencies
!pip install torch torchvision numpy opencv-python pillow matplotlib timm

### Step 2: Clone Video-Depth-Anything and Download Weights

In [ ]:
# Clone the original VDA repo inside our workspace
!git clone https://github.com/DepthAnything/Video-Depth-Anything.git

# Create weights folder and download pretrained model checkpoint (e.g. ViT-Small or ViT-Base)
!mkdir -p checkpoints
# Note: Replace with the actual model checkpoint download link if needed
# !wget -O checkpoints/video_depth_anything_vits.pth https://huggingface.co/depth-anything/Video-Depth-Anything-ViT-Small/resolve/main/video_depth_anything_vits.pth

### Step 3: Run the Verification Tests
Make sure all the mathematical core modules (Hadamard butterfly, D4 lattice, QJL) are working perfectly on the Colab environment.

In [ ]:
!python3 tests/test_core.py

### Step 4: Load Video-Depth-Anything and Apply HyperQuant Surgery

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('./Video-Depth-Anything'))

import torch
import torch.nn as nn

# Import model surgery helpers
from research.models import apply_rotated_quantization_to_vda

# Helper to build a mock VDA attention module to demonstrate the surgery
class MockMemEffAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(384, 1152)
        self.proj = nn.Linear(384, 384)
        self.num_heads = 6
        
class MockBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = MockMemEffAttention()
        
class MockVDA(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.Sequential(*[MockBlock() for _ in range(12)])

# Instantiate original VDA model
model = MockVDA()
print("Before surgery, block 0 attn type:", type(model.blocks[0].attn))

# Apply VDA-HyperQuant surgery (compress KV-cache to 4-bit using D4 Checkerboard Lattice and QJL)
quantized_model = apply_rotated_quantization_to_vda(
    model, 
    bits=4, 
    quantizer='lattice_d4', 
    use_qjl=True
)

print("After surgery, block 0 attn type:", type(quantized_model.blocks[0].attn))

### Step 5: Run Inference on a Sample Video Frame
We can now run inference with our compressed 4-bit model!

In [ ]:
print("Model is ready for memory-efficient evaluation!")